In [ ]:
# 1. Install Dependencies
# !pip install -U pip
# !pip install -U transformers peft bitsandbytes accelerate scikit-learn

# 2. Login to Hugging Face
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
# Ensure your secret is named 'HF_KEY' in the Add-ons menu
hf_token = user_secrets.get_secret("HF_KEY") 
login(token=hf_token)

print("Libraries installed and logged in successfully!")

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from torch.utils.data import DataLoader

# 1. Load Data
print("Loading dataset...")
dataset = load_dataset("ailsntua/QEvasion")

# 2. Prepare Labels
labels = sorted(dataset["train"].unique("clarity_label"))
num_labels = len(labels)
label2id = {l: i for i, l in enumerate(labels)}
# Map text labels to integers
dataset = dataset.map(lambda x: {"labels": label2id[x["clarity_label"]]})
# Remove the old text column to keep things clean
dataset = dataset.remove_columns(["clarity_label"])

# 3. Format Input Text
dataset = dataset.map(lambda x: {
    "text": f"Question: {x['interview_question']}\nAnswer: {x['interview_answer']}"
})

# 4. Tokenizer
model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token 

# 5. Tokenize (Consistent 1024 length)
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=True,
        max_length=1024 
    )

print("Tokenizing...")
encoded = dataset.map(tokenize_fn, batched=True)

# 6. Format Columns for PyTorch
keep_cols = ["input_ids", "attention_mask", "labels"]
encoded = encoded.remove_columns([c for c in encoded["train"].column_names if c not in keep_cols])
encoded.set_format("torch")

# 7. Create Loaders
# Batch size 4 + Grad Accumulation 4 = Effective Batch 16
batch_size = 4 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(encoded["train"], batch_size=batch_size, shuffle=True, collate_fn=data_collator)
valid_loader = DataLoader(encoded["test"], batch_size=batch_size, collate_fn=data_collator)

print(f"Data Ready. Train batches: {len(train_loader)}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# 1. Calculate Class Weights (Inverse Frequency)
train_labels = dataset["train"]["labels"]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)

# Convert to Tensor (Move to GPU later)
# We use standard float32 for the weights
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print(f"Computed Class Weights: {class_weights}")

# 2. Define the Focal Loss Class
class WeightedFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(WeightedFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        # Calculate Standard Cross Entropy (No reduction yet)
        ce_loss = F.cross_entropy(logits, targets, reduction='none', weight=self.alpha)
        
        # Calculate probabilities (pt)
        pt = torch.exp(-ce_loss)
        
        # Focal Term: (1 - pt)^gamma
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        return focal_loss.mean()

print("WeightedFocalLoss defined.")

In [ ]:
# --- BLOCK 4 REPLACEMENT ---

from transformers import BitsAndBytesConfig, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# 1. Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Load Base Model (FORCE TO GPU 0)
print(f"Loading {model_name}...")

# CHANGE IS HERE: device_map={'': 0} forces everything to GPU 0
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    num_labels=num_labels,
    device_map={'': 0}   # <--- FIX: Do not use "auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

# 3. Gradient Checkpointing
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# 4. LoRA Config (Rank 64)
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                    "gate_proj", "up_proj", "down_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# --- BLOCK 5 (CLEAN & SAFE) ---
import os
from tqdm import tqdm
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report

SAVE_PATH = "/kaggle/working/llama_focal_final"
os.makedirs(SAVE_PATH, exist_ok=True)

device = "cuda:0" # Everything is here now

# Define Loss (Make sure weights are on cuda:0)
loss_fct = WeightedFocalLoss(alpha=class_weights_tensor.to(device), gamma=2.0)

lr = 2e-4
num_epochs = 3 
gradient_accumulation_steps = 4 

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
num_training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=num_training_steps)

best_macro_f1 = 0

print("Starting Training (Single GPU Mode)...")

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    optimizer.zero_grad()
    
    for step, batch in enumerate(pbar):
        # Move inputs to cuda:0
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        
        # Loss
        logits = outputs.logits.view(-1, num_labels)
        targets = labels.view(-1)
        loss = loss_fct(logits, targets)
        
        loss = loss / gradient_accumulation_steps
        loss.backward()
        
        if (step + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * gradient_accumulation_steps
        pbar.set_postfix({'loss': loss.item() * gradient_accumulation_steps})

        # Safety Save
        if (step + 1) % 500 == 0:
            ckpt_path = os.path.join(SAVE_PATH, f"ckpt_ep{epoch}_step{step+1}")
            model.save_pretrained(ckpt_path)
            tokenizer.save_pretrained(ckpt_path)

    # --- Validation ---
    model.eval()
    preds, gts = [], []
    torch.cuda.empty_cache()
    
    with torch.no_grad():
        for batch in valid_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().tolist())
            gts.extend(batch['labels'].cpu().tolist())

    print(f"\nReport Epoch {epoch}:")
    print(classification_report(gts, preds, target_names=[str(l) for l in labels], digits=4))

    val_macro = f1_score(gts, preds, average='macro')
    if val_macro > best_macro_f1:
        best_macro_f1 = val_macro
        print(f"New Best! Saving...")
        model.save_pretrained(os.path.join(SAVE_PATH, "best_model"))
        tokenizer.save_pretrained(os.path.join(SAVE_PATH, "best_model"))